In [ ]:
!pip install transformers datasets evaluate rouge_score bert-score nltk sentencepiece -q
print('Done')


In [ ]:
import torch
import pandas as pd
import numpy as np
import evaluate
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from bert_score import score as bert_score
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


In [ ]:
test_df = pd.read_parquet('test-sci.parquet')

df_10 = test_df.sample(n=10, random_state=42)

articles             = df_10['document'].tolist()
reference_summaries  = df_10['summary'].tolist()

print('Articles:', len(articles))
print('Summaries:', len(reference_summaries))


In [ ]:
bart_tokenizer = AutoTokenizer.from_pretrained('facebook/bart-large-cnn')
bart_model     = AutoModelForSeq2SeqLM.from_pretrained('facebook/bart-large-cnn').to(device)
print('BART loaded')


In [ ]:
nli_model = pipeline(
    'text-classification',
    model='facebook/bart-large-mnli',
    device=0 if torch.cuda.is_available() else -1
)
print('NLI model loaded')


## Core Functions — NLI, Evidence Retrieval, Sentence Rewriting, Mitigation

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CACHE: avoids recomputing the same NLI label for the same
# (article, sentence) pair across faithfulness_score, mitigate_summary,
# hallucination_rate, and is_summary_hallucinated.
# Same checks, same model, same labels — just computed once each.
# ─────────────────────────────────────────────────────────────────
_label_cache = {}

def cached_label(article_id, chunks, sentence):
    key = (article_id, sentence)
    if key not in _label_cache:
        _label_cache[key] = get_best_label_across_chunks(chunks, sentence)
    return _label_cache[key]


# ─────────────────────────────────────────────────────────────────
# STEP 1 HELPER: Build overlapping chunks from an article
# ─────────────────────────────────────────────────────────────────
def get_chunks(article, chunk_size=400, overlap=50):
    words = article.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunks.append(' '.join(words[i:i + chunk_size]))
        if i + chunk_size >= len(words):
            break
    return chunks


# ─────────────────────────────────────────────────────────────────
# STEP 2 HELPER: NLI label for a single (premise, hypothesis) pair
# ─────────────────────────────────────────────────────────────────
def get_nli_label(premise, hypothesis):
    result = nli_model([{'text': premise, 'text_pair': hypothesis}], truncation=True)
    return result[0]['label'].lower()   # 'entailment', 'neutral', 'contradiction'


# ─────────────────────────────────────────────────────────────────
# STEP 3 HELPER: Best NLI label across all chunks for a sentence
# Priority: entailment > neutral > contradiction
# ─────────────────────────────────────────────────────────────────
def get_best_label_across_chunks(chunks, sentence):
    labels = [get_nli_label(chunk, sentence) for chunk in chunks]
    if 'entailment' in labels:
        return 'entailment'
    elif 'neutral' in labels:
        return 'neutral'
    else:
        return 'contradiction'


# ─────────────────────────────────────────────────────────────────
# STEP 4 HELPER: Faithfulness score (used to rank candidates)
# Uses cached_label so repeated sentences across candidates,
# and later reuse in mitigate_summary, don't re-run NLI.
# ─────────────────────────────────────────────────────────────────
def faithfulness_score(article_id, article, summary):
    sentences = sent_tokenize(str(summary))
    if not sentences:
        return 0.0
    chunks = get_chunks(article)
    entailed = 0
    for sent in sentences:
        if cached_label(article_id, chunks, sent) == 'entailment':
            entailed += 1
    return entailed / len(sentences)


# ─────────────────────────────────────────────────────────────────
# STEP 5 HELPER: Find most relevant evidence chunk
# ─────────────────────────────────────────────────────────────────
def get_best_evidence_chunk(article, sentence):
    chunks = get_chunks(article)
    sent_words = set(sentence.lower().split())
    best_chunk, best_overlap = chunks[0], -1
    for chunk in chunks:
        overlap = len(sent_words & set(chunk.lower().split()))
        if overlap > best_overlap:
            best_overlap = overlap
            best_chunk = chunk
    return best_chunk


# ─────────────────────────────────────────────────────────────────
# STEP 6 HELPER: Rewrite a hallucinated sentence using evidence
# ─────────────────────────────────────────────────────────────────
def rewrite_sentence_with_evidence(evidence_chunk, original_sentence, tokenizer, model):
    prompt = (
        f'Evidence: {evidence_chunk}\n'
        f'Rewrite the following sentence using only information from the evidence above.\n'
        f'Sentence: {original_sentence}'
    )
    inputs = tokenizer(
        prompt,
        max_length=1024,
        truncation=True,
        return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        output_ids = model.generate(
            inputs['input_ids'],
            max_new_tokens=80,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()


# ─────────────────────────────────────────────────────────────────
# MAIN MITIGATION FUNCTION
#
# Hallucination definition (updated):
#   Both 'contradiction' and 'neutral' are treated as hallucinated
#   and sent for rewriting.
#
# Label logic per sentence:
#   entailment   -> keep unchanged
#   neutral      -> rewrite using evidence
#                    -> if rewrite is entailed -> replace
#                    -> if rewrite fails  -> KEEP original (not factually wrong)
#   contradiction -> rewrite using evidence
#                    -> if rewrite is entailed/neutral -> replace
#                    -> if rewrite fails  -> REMOVE (factually wrong)
#
# article_id must be a stable, unique identifier per article (e.g. its
# index in the `articles` list) so the cache keys correctly.
# ─────────────────────────────────────────────────────────────────
def mitigate_summary(article_id, article, candidates, tokenizer, model):
    # Stage 1: pick best candidate by faithfulness score
    # (labels computed here are cached and reused below for the winner)
    scored = [(faithfulness_score(article_id, article, c), c) for c in candidates]
    best_score, best_candidate = max(scored, key=lambda x: x[0])

    sentences = sent_tokenize(best_candidate)
    chunks    = get_chunks(article)

    final_sentences  = []
    n_entailed       = 0
    n_neutral_regen  = 0   # neutral -> rewrite succeeded
    n_neutral_kept   = 0   # neutral -> rewrite failed -> kept as-is
    n_contra_regen   = 0   # contradiction -> rewrite succeeded
    n_contra_removed = 0   # contradiction -> rewrite failed -> removed

    for sent in sentences:
        # Reused from faithfulness_score's pass over this same candidate -
        # no re-run of NLI for sentences already labeled above.
        label = cached_label(article_id, chunks, sent)

        if label == 'entailment':
            final_sentences.append(sent)
            n_entailed += 1

        elif label == 'neutral':
            # Neutral is hallucinated -> attempt rewrite
            evidence  = get_best_evidence_chunk(article, sent)
            rewritten = rewrite_sentence_with_evidence(evidence, sent, tokenizer, model)
            rewrite_label = cached_label(article_id, chunks, rewritten)

            if rewrite_label == 'entailment':
                # Rewrite is now grounded -> replace
                final_sentences.append(rewritten)
                n_neutral_regen += 1
            else:
                # Rewrite not entailed -> keep original (neutral is not factually wrong)
                final_sentences.append(sent)
                n_neutral_kept += 1

        else:  # contradiction
            # Contradiction is hallucinated -> attempt rewrite
            evidence  = get_best_evidence_chunk(article, sent)
            rewritten = rewrite_sentence_with_evidence(evidence, sent, tokenizer, model)
            rewrite_label = cached_label(article_id, chunks, rewritten)

            if rewrite_label in ('entailment', 'neutral'):
                # Rewrite is acceptable -> replace
                final_sentences.append(rewritten)
                n_contra_regen += 1
            else:
                # Still contradictory after rewrite -> remove (factually wrong)
                n_contra_removed += 1

    mitigated = ' '.join(final_sentences)

    stats = {
        'total_sentences'  : len(sentences),
        'n_entailed'       : n_entailed,
        'n_neutral_regen'  : n_neutral_regen,
        'n_neutral_kept'   : n_neutral_kept,
        'n_contra_regen'   : n_contra_regen,
        'n_contra_removed' : n_contra_removed,
    }

    return mitigated, stats


# ─────────────────────────────────────────────────────────────────
# HALLUCINATION RATE (updated):
#   Both contradiction AND neutral count as hallucinated.
#   Score = (neutral + contradiction sentences) / total sentences
#
# Uses cached_label: for the mitigated summary, most sentences were
# already labeled inside mitigate_summary, so this is nearly free.
# For the *original* (pre-mitigation) summary, sentences are labeled
# here for the first time and the result is cached for is_summary_hallucinated().
# ─────────────────────────────────────────────────────────────────
def hallucination_rate(article_id, article, summary):
    sentences = sent_tokenize(str(summary))
    if not sentences:
        return 0.0
    chunks = get_chunks(article)
    hallucinated = sum(
        1 for sent in sentences
        if cached_label(article_id, chunks, sent) in ('contradiction', 'neutral')
    )
    return hallucinated / len(sentences)


print('All functions defined (with NLI label caching)')


## BART — Generate Candidates → Select Best → Rewrite Unsupported Sentences

In [ ]:
bart_original_summaries  = []
bart_mitigated_summaries = []
bart_stats_list          = []

for article_id, article in enumerate(tqdm(articles, desc='BART')):
    inputs = bart_tokenizer(
        article,
        max_length=1024,
        truncation=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        summary_ids = bart_model.generate(
            inputs['input_ids'],
            max_length=130,
            min_length=30,
            num_beams=8,
            num_return_sequences=4,
            early_stopping=True
        )

    candidates = [
        bart_tokenizer.decode(s, skip_special_tokens=True)
        for s in summary_ids
    ]

    # Original = first beam (highest beam score)
    bart_original_summaries.append(candidates[0])

    # Mitigated = best candidate + sentence-level rewriting
    mitigated, stats = mitigate_summary(article_id, article, candidates, bart_tokenizer, bart_model)
    bart_mitigated_summaries.append(mitigated)
    bart_stats_list.append(stats)

print(f'BART done. {len(bart_original_summaries)} originals, {len(bart_mitigated_summaries)} mitigated')


## T5 — Generate Candidates → Select Best → Rewrite Unsupported Sentences

In [ ]:
t5_tokenizer = AutoTokenizer.from_pretrained('t5-base')
t5_model     = AutoModelForSeq2SeqLM.from_pretrained('t5-base').to(device)
print('T5 loaded')


In [ ]:
t5_original_summaries  = []
t5_mitigated_summaries = []
t5_stats_list          = []

for article_id, article in enumerate(tqdm(articles, desc='T5')):
    inputs = t5_tokenizer(
        'summarize: ' + str(article),
        max_length=1024,
        truncation=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        summary_ids = t5_model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=180,
            min_length=60,
            num_beams=8,
            num_return_sequences=4,
            length_penalty=1.2,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    candidates = [
        t5_tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        for s in summary_ids
    ]

    t5_original_summaries.append(candidates[0])

    mitigated, stats = mitigate_summary(article_id, article, candidates, t5_tokenizer, t5_model)
    t5_mitigated_summaries.append(mitigated)
    t5_stats_list.append(stats)

print(f'T5 done. {len(t5_original_summaries)} originals, {len(t5_mitigated_summaries)} mitigated')


## Hallucination Rates — Before vs After Mitigation

In [ ]:
bart_hall_orig = [hallucination_rate(i, a, s) for i, (a, s) in enumerate(zip(articles, bart_original_summaries))]
bart_hall_mit  = [hallucination_rate(i, a, s) for i, (a, s) in enumerate(zip(articles, bart_mitigated_summaries))]
t5_hall_orig   = [hallucination_rate(i, a, s) for i, (a, s) in enumerate(zip(articles, t5_original_summaries))]
t5_hall_mit    = [hallucination_rate(i, a, s) for i, (a, s) in enumerate(zip(articles, t5_mitigated_summaries))]

print(f'BART  | Original hall. rate (neutral+contradiction): {np.mean(bart_hall_orig):.4f} | Mitigated: {np.mean(bart_hall_mit):.4f}')
print(f'T5    | Original hall. rate (neutral+contradiction): {np.mean(t5_hall_orig):.4f}   | Mitigated: {np.mean(t5_hall_mit):.4f}')


## Sentence-Level Regeneration Stats

In [ ]:
def summarise_stats(stats_list, model_name):
    total          = sum(s['total_sentences']  for s in stats_list)
    entailed       = sum(s['n_entailed']       for s in stats_list)
    neutral_regen  = sum(s['n_neutral_regen']  for s in stats_list)
    neutral_kept   = sum(s['n_neutral_kept']   for s in stats_list)
    contra_regen   = sum(s['n_contra_regen']   for s in stats_list)
    contra_removed = sum(s['n_contra_removed'] for s in stats_list)
    print(f'--- {model_name} ---')
    print(f'  Total sentences                                    : {total}')
    print(f'  Entailed (kept unchanged)                          : {entailed}  ({entailed/total*100:.1f}%)')
    print(f'  Neutral  → rewritten & now entailed                : {neutral_regen}  ({neutral_regen/total*100:.1f}%)')
    print(f'  Neutral  → rewrite not entailed → kept original    : {neutral_kept}  ({neutral_kept/total*100:.1f}%)')
    print(f'  Contradiction → rewritten & accepted               : {contra_regen}  ({contra_regen/total*100:.1f}%)')
    print(f'  Contradiction → rewrite failed → removed           : {contra_removed}  ({contra_removed/total*100:.1f}%)')

summarise_stats(bart_stats_list, 'BART')
print()
summarise_stats(t5_stats_list, 'T5')


## ROUGE — Before vs After Mitigation

In [ ]:
rouge = evaluate.load('rouge')

bart_rouge_orig = rouge.compute(predictions=bart_original_summaries,  references=reference_summaries)
bart_rouge_mit  = rouge.compute(predictions=bart_mitigated_summaries, references=reference_summaries)
t5_rouge_orig   = rouge.compute(predictions=t5_original_summaries,    references=reference_summaries)
t5_rouge_mit    = rouge.compute(predictions=t5_mitigated_summaries,   references=reference_summaries)

print('BART  ROUGE original : ', {k: round(v,4) for k,v in bart_rouge_orig.items()})
print('BART  ROUGE mitigated: ', {k: round(v,4) for k,v in bart_rouge_mit.items()})
print('T5    ROUGE original : ', {k: round(v,4) for k,v in t5_rouge_orig.items()})
print('T5    ROUGE mitigated: ', {k: round(v,4) for k,v in t5_rouge_mit.items()})


## BERTScore — Before vs After Mitigation

In [ ]:
_, _, F1_bart_orig = bert_score(bart_original_summaries,  reference_summaries, lang='en', verbose=False)
_, _, F1_bart_mit  = bert_score(bart_mitigated_summaries, reference_summaries, lang='en', verbose=False)
_, _, F1_t5_orig   = bert_score(t5_original_summaries,    reference_summaries, lang='en', verbose=False)
_, _, F1_t5_mit    = bert_score(t5_mitigated_summaries,   reference_summaries, lang='en', verbose=False)

print('BART  BERTScore original : ', round(F1_bart_orig.mean().item(), 4))
print('BART  BERTScore mitigated: ', round(F1_bart_mit.mean().item(),  4))
print('T5    BERTScore original : ', round(F1_t5_orig.mean().item(),   4))
print('T5    BERTScore mitigated: ', round(F1_t5_mit.mean().item(),    4))


## Final Results Table

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Summary-level hallucination score helper
# A summary is hallucinated if ANY sentence is neutral OR contradiction
# ─────────────────────────────────────────────────────────────────
def is_summary_hallucinated(article_id, article, summary):
    sentences = sent_tokenize(str(summary))
    if not sentences:
        return False
    chunks = get_chunks(article)
    for sent in sentences:
        # Reuses labels already cached by hallucination_rate() above for
        # this exact (article_id, sentence) -- no re-run of NLI here.
        if cached_label(article_id, chunks, sent) in ('contradiction', 'neutral'):
            return True
    return False

def compute_summary_level_score(articles, summaries, desc=''):
    total        = len(summaries)
    hallucinated = 0
    for article_id, (article, summary) in enumerate(tqdm(list(zip(articles, summaries)), total=total,
                                 desc=f'Summary-level [{desc}]')):
        if is_summary_hallucinated(article_id, article, summary):
            hallucinated += 1
    return round(hallucinated / total, 4) if total > 0 else 0.0

print('Computing summary-level hallucination scores...')
bart_summ_orig = compute_summary_level_score(articles, bart_original_summaries,  desc='BART Original')
bart_summ_mit  = compute_summary_level_score(articles, bart_mitigated_summaries, desc='BART Mitigated')
t5_summ_orig   = compute_summary_level_score(articles, t5_original_summaries,    desc='T5 Original')
t5_summ_mit    = compute_summary_level_score(articles, t5_mitigated_summaries,   desc='T5 Mitigated')

final_results = pd.DataFrame({
    'Model': [
        'BART (original)', 'BART (mitigated)',
        'T5 (original)',   'T5 (mitigated)'
    ],
    'ROUGE-1': [
        bart_rouge_orig['rouge1'], bart_rouge_mit['rouge1'],
        t5_rouge_orig['rouge1'],   t5_rouge_mit['rouge1']
    ],
    'ROUGE-2': [
        bart_rouge_orig['rouge2'], bart_rouge_mit['rouge2'],
        t5_rouge_orig['rouge2'],   t5_rouge_mit['rouge2']
    ],
    'ROUGE-L': [
        bart_rouge_orig['rougeL'], bart_rouge_mit['rougeL'],
        t5_rouge_orig['rougeL'],   t5_rouge_mit['rougeL']
    ],
    'BERTScore F1': [
        round(F1_bart_orig.mean().item(), 4), round(F1_bart_mit.mean().item(), 4),
        round(F1_t5_orig.mean().item(),   4), round(F1_t5_mit.mean().item(),   4)
    ],
    'Summary-Level Hall. Score (neutral+contra)': [
        bart_summ_orig, bart_summ_mit,
        t5_summ_orig,   t5_summ_mit
    ],
    'Neutral Rewritten': [
        '-', sum(s['n_neutral_regen']  for s in bart_stats_list),
        '-', sum(s['n_neutral_regen']  for s in t5_stats_list)
    ],
    'Contradiction Rewritten': [
        '-', sum(s['n_contra_regen']   for s in bart_stats_list),
        '-', sum(s['n_contra_regen']   for s in t5_stats_list)
    ],
    'Removed (contradiction)': [
        '-', sum(s['n_contra_removed'] for s in bart_stats_list),
        '-', sum(s['n_contra_removed'] for s in t5_stats_list)
    ]
})

print(final_results.to_string(index=False))
final_results


In [ ]:
output_df = pd.DataFrame({
    'Article'           : articles,
    'Reference'         : reference_summaries,
    'BART Original'     : bart_original_summaries,
    'BART Mitigated'    : bart_mitigated_summaries,
    'BART Hall Orig'    : bart_hall_orig,
    'BART Hall Mit'     : bart_hall_mit,
    'T5 Original'       : t5_original_summaries,
    'T5 Mitigated'      : t5_mitigated_summaries,
    'T5 Hall Orig'      : t5_hall_orig,
    'T5 Hall Mit'       : t5_hall_mit,
})

output_df.to_csv('bart_t5_mitigated_results.csv', index=False)
print('Saved to bart_t5_mitigated_results.csv')
